In [ ]:
!pip install transformers torch sentence-transformers langchain langchain-community langchain-cohere faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 4.3 MB/s eta 0:00:00


In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA
from langchain_community.llms import HuggingFacePipeline
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_cohere import ChatCohere

In [ ]:
doc = PyPDFLoader("./Introduction_to_RAG.pdf")
documents = [pages for pages in doc.load()]

In [ ]:
model_hugging = "google/flan-t5-small"
tokenizer_hugging = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

pipe = pipeline("text-generation", model=model_hugging, tokenizer=tokenizer_hugging, device=-1, max_new_tokens=1000)

llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'AXK1ForCausalLM', 'AXK2ForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'Cohere2MoeForCausalLM', 'CohereCompassForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-V2")

/tmp/ipykernel_1123/2320153371.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-V2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
vector_store = FAISS.from_documents(documents=documents, embedding=embeddings)

retriever = vector_store.as_retriever()

qa = RetrievalQA.from_chain_type(llm=llm, retriever=retriever, return_source_documents=True)

query = "what is a vector database"
output = qa.invoke(query)

answer_retrieved = output["result"]
print(answer_retrieved)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1184 > 512). Running this sequence through the model will result in indexing errors
[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Structured Data - SQL Databases-NoSQL 
Databases- Knowledge 
Graphs (e.g., Neo4j, 
Wikidata) 
Retrieve structured facts 
and relational data 
Unstructured Documents - Internal PDFs, Word 
Docs-Company Policies- 
Reports 
Fetch content from raw, 
unformatted or semi-
formatted documents 
Enterprise Systems - CRM systems (e.g., 
Salesforce)-Ticketing 
systems (e.g., Zendesk)-
SharePoint, Confluence 
Pull insights from 
internal communication 
or customer data 
Developer Knowledge 
Bases 
- Stack Overflow- GitHub-
Docs repositories 
Supplement coding-
related queries 
Scientific/Academic 
Sources 
- PubMed-arXiv-
ResearchGate 
Retrieve credible 
scientific research for 
technical domains 
Communication Archives - Slack/Teams logs-
Emails 
Capture organization-
specific Q&A and 
discussions 
 
Key Use Cases of RAG 
1. Customer S

In [ ]:
prompt = f'''Evaluate the following answer for correctness, hallucinations and completeness. If the answer is weak, suggest improvements under the heading "feedack" at the end
question: {query}
answer_retrieved: {answer_retrieved}
format the answer properly'''

cohere_llm = ChatCohere(user_agent="langchain", cohere_api_key="API_KEY")

feedback = print(cohere_llm.invoke(prompt).content)

**Evaluation of the Retrieved Answer:**

The retrieved answer is **weak** and **incomplete**. It fails to directly and clearly define what a vector database is. Instead, it provides a list of categories, examples, and use cases related to various types of databases and systems, including vector databases, but does not explain the concept itself. The answer also includes irrelevant information about Retrieval-Augmented Generation (RAG) and its use cases, which distracts from the core question.

**Key Issues:**
1. **Lack of Definition:** The answer does not define a vector database.
2. **Relevance:** Much of the content is unrelated to the question.
3. **Hallucinations:** There are no hallucinations, but the answer is incomplete and off-topic.
4. **Formatting:** The answer is poorly formatted and lacks clarity.

**Correct and Complete Answer:**

A **vector database** is a specialized type of database designed to store, index, and query high-dimensional vector embeddings efficiently. Thes

In [ ]:
query_updated = f'''{query} use the following feedback: {feedback} to answer the query'''

result_updated = qa.invoke(query_updated)

print(result_updated['result'])

[transformers] Both `max_new_tokens` (=1000) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

Structured Data - SQL Databases-NoSQL 
Databases- Knowledge 
Graphs (e.g., Neo4j, 
Wikidata) 
Retrieve structured facts 
and relational data 
Unstructured Documents - Internal PDFs, Word 
Docs-Company Policies- 
Reports 
Fetch content from raw, 
unformatted or semi-
formatted documents 
Enterprise Systems - CRM systems (e.g., 
Salesforce)-Ticketing 
systems (e.g., Zendesk)-
SharePoint, Confluence 
Pull insights from 
internal communication 
or customer data 
Developer Knowledge 
Bases 
- Stack Overflow- GitHub-
Docs repositories 
Supplement coding-
related queries 
Scientific/Academic 
Sources 
- PubMed-arXiv-
ResearchGate 
Retrieve credible 
scientific research for 
technical domains 
Communication Archives - Slack/Teams logs-
Emails 
Capture organization-
specific Q&A and 
discussions 
 
Key Use Cases of RAG 
1. Customer S